# Convert Original Diagram Annotations

This notebook converts the original COCO-style annotations from the public
`handwritten-diagram-datasets` repository into the per-image graph JSON format
used by the paper.

The routing and node-linking logic is preserved from the final clean converter.
The notebook:

- reads the original `fa`, `fca`, and `fcb` dataset families;
- converts node boxes, arrow endpoints, directed links, and routed arrow paths;
- scales all annotation coordinates to a common **1024 × 1024** canvas;
- writes one JSON annotation per image;
- optionally writes visual routing overlays;
- produces split-level and repository-level conversion summaries.

The original images are **not** copied or resized here. Run the separate image
resizing notebook after this conversion so that the images and annotations share
the same 1024 × 1024 coordinate system.

Expected source layout for each family:

```text
<source_root>/
├── fa/
│   ├── train.json
│   ├── train/
│   ├── val.json       # when provided by the source dataset
│   ├── val/
│   ├── test.json
│   └── test/
├── fca/
└── fcb/
```


In [ ]:
# ============================================================
# USER SETTINGS — FILL THE PATHS BEFORE RUNNING
# ============================================================

# Folder containing the original dataset family folders: fa/, fca/, and fcb/
SOURCE_DATASETS_ROOT = ""

# Folder in which the converted per-image JSON annotations will be created
OUTPUT_ANNOTATIONS_ROOT = ""

# Dataset families to convert. Remove a name when only a subset is required.
FAMILIES_TO_CONVERT = ["fa", "fca", "fcb"]

# Use None to automatically discover all valid splits in each family.
# Example for a restricted run: ["train", "val", "test"]
SPLITS_TO_CONVERT = None

# All output annotations are expressed in this coordinate system.
TARGET_WIDTH = 1024
TARGET_HEIGHT = 1024

# Conversion behavior
ROUTING_MODE = "auto"       # auto, flowchart, fa, straight, or waypoint
TRACE_FROM_IMAGE = True
SAVE_OVERLAYS = False       # optional visual checks; can require substantial space
SAVE_ROUTING_DEBUG = False  # adds routing diagnostics inside every arrow record

# Notebook-safe thread workers. Set to 1 for a fully sequential run.
WORKERS = 4


## 1. Imports and dependency check

Required packages are `numpy`, `opencv-python` (or
`opencv-python-headless`), and `scikit-image`. The converter can fall back
without `scikit-image`, but installing it preserves the intended skeletonization
route.


In [ ]:
from __future__ import annotations

import json
import math
import heapq
import os
from pathlib import Path
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np

try:
    from skimage.morphology import skeletonize
    HAS_SKIMAGE = True
except Exception:
    HAS_SKIMAGE = False

print(f"OpenCV version: {cv2.__version__}")
print(f"scikit-image skeletonization available: {HAS_SKIMAGE}")


## 2. Geometry and common routing utilities

These functions operate in the original image coordinate system during routing.
Coordinates are scaled to the final 1024 × 1024 canvas only when the per-image
JSON record is written.


In [ ]:
# ============================================================
# Geometry helpers
# ============================================================

def pt(x, y):
    return {"x": float(x), "y": float(y)}


def bbox_to_xyxy(bbox):
    x, y, w, h = bbox
    return int(x), int(y), int(x + w), int(y + h)


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def scale_point(p, sx, sy):
    return {"x": float(p["x"]) * sx, "y": float(p["y"]) * sy}


def scale_bbox(bbox, sx, sy):
    x, y, w, h = bbox
    return [
        int(round(x * sx)),
        int(round(y * sy)),
        int(round(w * sx)),
        int(round(h * sy)),
    ]


def scale_path(path, sx, sy):
    return [scale_point(p, sx, sy) for p in path]


def straight_line_path(p0, p1):
    x0, y0 = p0["x"], p0["y"]
    x1, y1 = p1["x"], p1["y"]

    n = int(max(abs(x1 - x0), abs(y1 - y0))) + 1
    if n <= 1:
        return [pt(x0, y0)]

    xs = np.linspace(x0, x1, n)
    ys = np.linspace(y0, y1, n)

    return [pt(x, y) for x, y in zip(xs, ys)]


def polyline_path(points):
    if not points:
        return []

    out = []
    for i in range(len(points) - 1):
        p0 = pt(points[i][0], points[i][1])
        p1 = pt(points[i + 1][0], points[i + 1][1])

        seg = straight_line_path(p0, p1)

        if i > 0 and seg:
            seg = seg[1:]

        out.extend(seg)

    return out


def waypoint_path(arrow_ann, tail, head):
    waypoints = arrow_ann.get("waypoints", None)

    if waypoints and len(waypoints) >= 2:
        return polyline_path(waypoints)

    return straight_line_path(tail, head)


def sample_equidistant_points(path, count=40):
    if not path or len(path) < 2:
        return []

    coords = np.array([[p["x"], p["y"]] for p in path], dtype=np.float32)
    seg = np.diff(coords, axis=0)
    seg_len = np.sqrt((seg ** 2).sum(axis=1))
    cum = np.concatenate([[0], np.cumsum(seg_len)])

    total = cum[-1]
    if total <= 1e-6:
        return []

    samples = []
    targets = np.linspace(0, total, count + 2)[1:-1]

    for i, t in enumerate(targets, start=1):
        idx = np.searchsorted(cum, t) - 1
        idx = clamp(idx, 0, len(seg_len) - 1)

        alpha = 0 if seg_len[idx] <= 1e-6 else (t - cum[idx]) / seg_len[idx]
        xy = coords[idx] * (1 - alpha) + coords[idx + 1] * alpha

        samples.append({
            "x": int(round(xy[0])),
            "y": int(round(xy[1])),
            "label": f"E{i}",
        })

    return samples


def bbox_center(bbox):
    x, y, w, h = bbox
    return np.array([x + w / 2.0, y + h / 2.0], dtype=np.float32)


In [ ]:
# ============================================================
# Shared routing helpers
# ============================================================

def make_disk_mask(shape, cx, cy, r):
    h, w = shape
    yy, xx = np.ogrid[:h, :w]
    return (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2


def draw_polyline_mask(shape, points, thickness=35):
    mask = np.zeros(shape, dtype=np.uint8)

    if len(points) < 2:
        return mask.astype(bool)

    pts = np.array(
        [[int(round(p[0])), int(round(p[1]))] for p in points],
        dtype=np.int32,
    )

    cv2.polylines(mask, [pts], False, 255, thickness=thickness)
    return mask > 0


def nearest_allowed_pixel(allowed, target_xy, max_radius=70):
    tx, ty = target_xy
    h, w = allowed.shape

    tx = clamp(int(round(tx)), 0, w - 1)
    ty = clamp(int(round(ty)), 0, h - 1)

    if allowed[ty, tx]:
        return tx, ty

    ys, xs = np.where(allowed)
    if len(xs) == 0:
        return None

    d2 = (xs - tx) ** 2 + (ys - ty) ** 2
    idx = int(np.argmin(d2))

    if max_radius is not None and math.sqrt(float(d2[idx])) > max_radius:
        return None

    return int(xs[idx]), int(ys[idx])


def nearest_mask_pixel(mask, target_xy, max_radius=None):
    return nearest_allowed_pixel(mask, target_xy, max_radius=max_radius)


def weighted_dijkstra_path(cost, allowed, start, goal):
    h, w = cost.shape
    sx, sy = start
    gx, gy = goal

    dist = np.full((h, w), np.inf, dtype=np.float32)
    visited = np.zeros((h, w), dtype=bool)
    parent = {}

    dist[sy, sx] = 0.0
    heap = [(0.0, sx, sy)]

    nbrs = [
        (-1, -1, 1.414), (0, -1, 1.0), (1, -1, 1.414),
        (-1,  0, 1.0),                  (1,  0, 1.0),
        (-1,  1, 1.414), (0,  1, 1.0),  (1,  1, 1.414),
    ]

    while heap:
        d, x, y = heapq.heappop(heap)

        if visited[y, x]:
            continue

        visited[y, x] = True

        if (x, y) == (gx, gy):
            break

        for dx, dy, step_cost in nbrs:
            nx, ny = x + dx, y + dy

            if nx < 0 or ny < 0 or nx >= w or ny >= h:
                continue

            if not allowed[ny, nx]:
                continue

            nd = d + step_cost * float(cost[ny, nx])

            if nd < dist[ny, nx]:
                dist[ny, nx] = nd
                parent[(nx, ny)] = (x, y)
                heapq.heappush(heap, (nd, nx, ny))

    if not np.isfinite(dist[gy, gx]):
        return None

    path = []
    cur = (gx, gy)

    while cur != (sx, sy):
        path.append(cur)
        cur = parent.get(cur)

        if cur is None:
            return None

    path.append((sx, sy))
    path.reverse()

    return path


def shortest_path_on_binary_mask(mask, start, goal):
    h, w = mask.shape
    sx, sy = start
    gx, gy = goal

    if sx < 0 or sy < 0 or gx < 0 or gy < 0 or sx >= w or gx >= w or sy >= h or gy >= h:
        return []

    if not mask[sy, sx] or not mask[gy, gx]:
        return []

    q = deque([(sx, sy)])
    visited = np.zeros((h, w), dtype=np.uint8)
    parent = {}

    visited[sy, sx] = 1

    nbrs = [
        (-1, -1), (0, -1), (1, -1),
        (-1,  0),          (1,  0),
        (-1,  1), (0,  1), (1,  1),
    ]

    found = False

    while q:
        x, y = q.popleft()

        if (x, y) == (gx, gy):
            found = True
            break

        for dx, dy in nbrs:
            nx, ny = x + dx, y + dy

            if nx < 0 or ny < 0 or nx >= w or ny >= h:
                continue

            if not mask[ny, nx]:
                continue

            if visited[ny, nx]:
                continue

            visited[ny, nx] = 1
            parent[(nx, ny)] = (x, y)
            q.append((nx, ny))

    if not found:
        return []

    path = []
    cur = (gx, gy)

    while True:
        path.append(cur)

        if cur == (sx, sy):
            break

        cur = parent[cur]

    path.reverse()
    return path


def dilate_bool_mask(mask, k=3, iterations=1):
    kernel = np.ones((k, k), np.uint8)
    out = cv2.dilate(mask.astype(np.uint8), kernel, iterations=iterations)
    return out > 0


def localize_points(points, ox, oy):
    return [[p[0] - ox, p[1] - oy] for p in points]


def bbox_intersects(b1, b2):
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2

    return not (
        x1 + w1 < x2 or x2 + w2 < x1 or
        y1 + h1 < y2 or y2 + h2 < y1
    )


def binarize_and_skeletonize_for_routing(gray):
    _, th = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )

    kernel = np.ones((3, 3), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=1)

    if HAS_SKIMAGE:
        skel = skeletonize(th > 0)
        return skel.astype(bool)

    return th > 0


## 3. Flowchart arrow routing

Flowchart connectors use their annotated waypoint corridors together with the
image evidence to recover a dense ordered path.


In [ ]:
# ============================================================
# Flowchart routing: uses explicit waypoints
# ============================================================

def extract_flowchart_path(
    image,
    arrow_ann,
    blocking_annotations,
    tail,
    head,
    pad=12,
    ink_threshold=180,
    endpoint_keep_radius=45,
    node_forbid_margin=10,
    corridor_thickness=45,
    allow_white_bridge=True,
):
    h, w = image.shape[:2]

    waypoints = arrow_ann.get("waypoints", None)
    guide_points = (
        waypoints
        if waypoints and len(waypoints) >= 2
        else [[tail["x"], tail["y"]], [head["x"], head["y"]]]
    )

    xs = [p[0] for p in guide_points] + [tail["x"], head["x"]]
    ys = [p[1] for p in guide_points] + [tail["y"], head["y"]]

    x1p = clamp(int(min(xs)) - corridor_thickness - pad, 0, w - 1)
    y1p = clamp(int(min(ys)) - corridor_thickness - pad, 0, h - 1)
    x2p = clamp(int(max(xs)) + corridor_thickness + pad, 0, w - 1)
    y2p = clamp(int(max(ys)) + corridor_thickness + pad, 0, h - 1)

    crop = image[y1p:y2p + 1, x1p:x2p + 1]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    ch, cw = gray.shape
    local_waypoints = localize_points(guide_points, x1p, y1p)

    corridor = draw_polyline_mask(
        (ch, cw),
        local_waypoints,
        thickness=corridor_thickness,
    )

    ink = gray < ink_threshold
    ink_u8 = ink.astype(np.uint8) * 255
    kernel = np.ones((3, 3), np.uint8)
    ink_u8 = cv2.morphologyEx(ink_u8, cv2.MORPH_CLOSE, kernel, iterations=1)
    ink = ink_u8 > 0

    allowed = corridor.copy()

    for ann in blocking_annotations:
        bx, by, bw, bh = ann["bbox"]

        lx1 = clamp(bx - x1p - node_forbid_margin, 0, cw - 1)
        ly1 = clamp(by - y1p - node_forbid_margin, 0, ch - 1)
        lx2 = clamp(bx + bw - x1p + node_forbid_margin, 0, cw - 1)
        ly2 = clamp(by + bh - y1p + node_forbid_margin, 0, ch - 1)

        region = np.zeros((ch, cw), dtype=bool)
        region[ly1:ly2 + 1, lx1:lx2 + 1] = True

        allowed[region & corridor] = False

    local_tail = (
        int(round(tail["x"] - x1p)),
        int(round(tail["y"] - y1p)),
    )

    local_head = (
        int(round(head["x"] - x1p)),
        int(round(head["y"] - y1p)),
    )

    for cx, cy in [local_tail, local_head]:
        cx = clamp(cx, 0, cw - 1)
        cy = clamp(cy, 0, ch - 1)

        disk = make_disk_mask((ch, cw), cx, cy, endpoint_keep_radius)
        allowed[disk & corridor] = True

    if allow_white_bridge:
        ink_u8 = ink.astype(np.uint8)

        dist_to_ink = cv2.distanceTransform(
            (1 - ink_u8).astype(np.uint8),
            cv2.DIST_L2,
            3,
        )

        cost = 1.0 + 8.0 * dist_to_ink
        cost[ink] = 1.0
    else:
        allowed &= ink
        cost = np.ones((ch, cw), dtype=np.float32)

    guide_line = draw_polyline_mask(
        (ch, cw),
        local_waypoints,
        thickness=3,
    ).astype(np.uint8)

    dist_to_guide = cv2.distanceTransform(
        (1 - guide_line).astype(np.uint8),
        cv2.DIST_L2,
        3,
    )

    cost += 0.35 * dist_to_guide
    cost[~allowed] = 1e9

    start = nearest_allowed_pixel(allowed, local_tail, max_radius=70)
    goal = nearest_allowed_pixel(allowed, local_head, max_radius=70)

    if start is None or goal is None:
        return waypoint_path(arrow_ann, tail, head)

    path_xy = weighted_dijkstra_path(cost, allowed, start, goal)

    if not path_xy or len(path_xy) < 2:
        return waypoint_path(arrow_ann, tail, head)

    return [pt(x + x1p, y + y1p) for x, y in path_xy]


## 4. Finite-automata and self-loop routing

Finite-automata arrows use ellipse-aware node blocking, adaptive retries, and a
specialized self-loop router. This section preserves the final clean routing
logic used to create the project annotations.


In [ ]:
# ============================================================
# FA routing: ellipse-aware node blocking + self-loop midpoint router
# ============================================================

FA_RETRY_CONFIGS = [
    {"node_ellipse_margin": 10, "endpoint_keep_radius": 38, "corridor_thickness": 65},
    {"node_ellipse_margin": 14, "endpoint_keep_radius": 30, "corridor_thickness": 75},
    {"node_ellipse_margin": 18, "endpoint_keep_radius": 24, "corridor_thickness": 90},
    {"node_ellipse_margin": 22, "endpoint_keep_radius": 20, "corridor_thickness": 105},
    {"node_ellipse_margin": 26, "endpoint_keep_radius": 16, "corridor_thickness": 120},
]


def ellipse_mask_for_bbox(shape, bbox, ox, oy, margin=8):
    h, w = shape
    x, y, bw, bh = bbox

    cx = x + bw / 2.0 - ox
    cy = y + bh / 2.0 - oy
    rx = max(1.0, bw / 2.0 + margin)
    ry = max(1.0, bh / 2.0 + margin)

    yy, xx = np.ogrid[:h, :w]
    return ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1.0


def path_inside_node_ratio(path, node_bbox, margin=4):
    if not path:
        return 1.0

    x, y, w, h = node_bbox
    cx = x + w / 2.0
    cy = y + h / 2.0
    rx = max(1.0, w / 2.0 + margin)
    ry = max(1.0, h / 2.0 + margin)

    inside = 0
    for p in path:
        val = ((p["x"] - cx) / rx) ** 2 + ((p["y"] - cy) / ry) ** 2
        if val <= 1.0:
            inside += 1

    return inside / max(len(path), 1)


def path_inside_node_ratio_strict(path, node_bbox, margin=10):
    return path_inside_node_ratio(path, node_bbox, margin=margin)


def find_node_by_source_id(node_anns, source_id):
    if source_id is None:
        return None

    for ann in node_anns:
        if ann.get("id") == source_id or ann.get("aid") == source_id:
            return ann

    return None


def is_same_node(prev_node, next_node):
    if prev_node is None or next_node is None:
        return False

    return (
        prev_node.get("id") == next_node.get("id") or
        prev_node.get("aid") == next_node.get("aid")
    )


def is_self_loop_arrow(arrow_ann, node_annotations):
    prev_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_prev"))
    next_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_next"))
    return is_same_node(prev_node, next_node), prev_node, next_node


def adaptive_endpoint_radius_for_loop(tail, head, default_radius):
    dx = tail["x"] - head["x"]
    dy = tail["y"] - head["y"]
    d = math.sqrt(dx * dx + dy * dy)

    # Prevent tail/head restore disks from merging into one tunnel.
    return int(max(10, min(default_radius, 0.32 * d)))


def synthetic_fa_guide_points(arrow_ann, tail, head, prev_node=None, next_node=None):
    same_node = is_same_node(prev_node, next_node)

    ax, ay, aw, ah = arrow_ann["bbox"]

    if not same_node or prev_node is None:
        return [[tail["x"], tail["y"]], [head["x"], head["y"]]]

    nc = bbox_center(prev_node["bbox"])
    ac = bbox_center(arrow_ann["bbox"])
    direction = ac - nc

    if abs(direction[0]) >= abs(direction[1]):
        # Left/right self-loop.
        side_x = ax if direction[0] < 0 else ax + aw
        mid1 = [side_x, ay + ah * 0.15]
        mid2 = [side_x, ay + ah * 0.85]
    else:
        # Top/bottom self-loop.
        side_y = ay if direction[1] < 0 else ay + ah
        mid1 = [ax + aw * 0.15, side_y]
        mid2 = [ax + aw * 0.85, side_y]

    return [
        [tail["x"], tail["y"]],
        mid1,
        mid2,
        [head["x"], head["y"]],
    ]



def path_length_pixels(path):
    if not path or len(path) < 2:
        return 0.0

    total = 0.0
    for p0, p1 in zip(path[:-1], path[1:]):
        total += math.hypot(p1["x"] - p0["x"], p1["y"] - p0["y"])

    return total


def synthetic_fa_guide_points_for_side(arrow_ann, tail, head, side):
    """
    Explicit self-loop guide for one chosen side of the arrow bbox.
    This is only a probe, not the final path.
    """

    ax, ay, aw, ah = arrow_ann["bbox"]

    if side == "left":
        mid1 = [ax, ay + ah * 0.15]
        mid2 = [ax, ay + ah * 0.85]
    elif side == "right":
        mid1 = [ax + aw, ay + ah * 0.15]
        mid2 = [ax + aw, ay + ah * 0.85]
    elif side == "top":
        mid1 = [ax + aw * 0.15, ay]
        mid2 = [ax + aw * 0.85, ay]
    elif side == "bottom":
        mid1 = [ax + aw * 0.15, ay + ah]
        mid2 = [ax + aw * 0.85, ay + ah]
    else:
        raise ValueError(f"Unknown loop side: {side}")

    return [
        [tail["x"], tail["y"]],
        mid1,
        mid2,
        [head["x"], head["y"]],
    ]


def ordered_loop_sides_from_geometry(arrow_ann, loop_node):
    """
    Returns likely loop sides in priority order.
    The first one is based on arrow bbox center relative to the node center,
    but we still try all sides because some FA annotations are tight/ugly.
    """

    ax, ay, aw, ah = arrow_ann["bbox"]
    ac = bbox_center(arrow_ann["bbox"])
    nc = bbox_center(loop_node["bbox"])
    direction = ac - nc

    if abs(direction[0]) >= abs(direction[1]):
        primary = "left" if direction[0] < 0 else "right"
    else:
        primary = "top" if direction[1] < 0 else "bottom"

    all_sides = ["top", "right", "bottom", "left"]
    ordered = [primary] + [s for s in all_sides if s != primary]
    return ordered


def choose_midpoint_on_real_loop_ink(
    route_mask,
    guide_band,
    arrow_region,
    start,
    goal,
    synthetic_mid_target,
    node_bbox,
    crop_origin,
    endpoint_exclusion_radius=18,
):
    """
    Selects a real skeleton pixel on the loop. The synthetic guide is used
    only to search the right region. The actual selected point must be ink.

    Scoring strongly prefers points far away from the node center. This avoids
    choosing a short bridge near the node boundary.
    """

    ch, cw = route_mask.shape
    x1p, y1p = crop_origin

    endpoint_exclusion = (
        make_disk_mask((ch, cw), start[0], start[1], endpoint_exclusion_radius) |
        make_disk_mask((ch, cw), goal[0], goal[1], endpoint_exclusion_radius)
    )

    candidates = route_mask & guide_band & arrow_region & (~endpoint_exclusion)

    # If arrow_region is too strict, relax to guide_band.
    if not candidates.any():
        candidates = route_mask & guide_band & (~endpoint_exclusion)

    if not candidates.any():
        return None

    ys, xs = np.where(candidates)
    pts = np.stack([xs, ys], axis=1).astype(np.float32)

    node_center_global = bbox_center(node_bbox)
    node_center = np.array(
        [node_center_global[0] - x1p, node_center_global[1] - y1p],
        dtype=np.float32,
    )

    mid_target = np.array(synthetic_mid_target, dtype=np.float32)

    dist_from_node = np.linalg.norm(pts - node_center[None, :], axis=1)
    dist_to_mid = np.linalg.norm(pts - mid_target[None, :], axis=1)

    # Normalize to avoid one term numerically bullying the other.
    dfn = dist_from_node / (dist_from_node.max() + 1e-6)
    dtm = dist_to_mid / (dist_to_mid.max() + 1e-6)

    # Main rule: pick real ink farthest from node, but still near synthetic side.
    score = 2.0 * dfn - 0.45 * dtm

    best_idx = int(np.argmax(score))
    return int(pts[best_idx, 0]), int(pts[best_idx, 1])




def make_ink_mask_for_weighted_loop(gray):
    """
    Robust black-ink mask. Otsu handles writer/style variation better than
    one fixed threshold for these FA images.
    """
    _, th = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )

    kernel = np.ones((3, 3), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=1)
    return th > 0


def weighted_route_forced_through_midpoint(
    gray,
    allowed_region,
    node_core,
    local_guide,
    start,
    mid,
    goal,
    extra_block_mask=None,
):
    """
    Fallback for stubborn FA self-loops.

    The midpoint router first tries pure skeleton BFS. That is clean but brittle:
    if a loop skeleton breaks near the state boundary, BFS fails.

    This fallback still forces tail -> midpoint -> head, but uses weighted routing:
    - black ink is cheap
    - near-ink white pixels are allowed but expensive
    - node core is forbidden
    - corridor/arrow region constrains the search
    """

    ch, cw = gray.shape

    ink = make_ink_mask_for_weighted_loop(gray)

    allowed = allowed_region.copy()
    allowed &= ~node_core

    if extra_block_mask is not None:
        allowed &= ~extra_block_mask

    # Make sure selected points are legal. They are already near real ink, but
    # tiny masking artifacts can otherwise invalidate them.
    for p in [start, mid, goal]:
        if p is None:
            return None, {
                "weighted_midpoint_used": False,
                "reason": "missing_start_mid_or_goal",
            }

        x, y = p
        if 0 <= x < cw and 0 <= y < ch:
            allowed[y, x] = True

    ink_u8 = ink.astype(np.uint8)
    dist_to_ink = cv2.distanceTransform(
        (1 - ink_u8).astype(np.uint8),
        cv2.DIST_L2,
        3,
    )

    # Cheap on ink, expensive away from ink.
    cost = 1.0 + 9.0 * dist_to_ink
    cost[ink] = 1.0

    # Prefer the synthetic side, but do not force it blindly.
    guide_line = draw_polyline_mask(
        (ch, cw),
        local_guide,
        thickness=3,
    ).astype(np.uint8)

    dist_to_guide = cv2.distanceTransform(
        (1 - guide_line).astype(np.uint8),
        cv2.DIST_L2,
        3,
    )

    cost += 0.22 * dist_to_guide

    # Repel from node boundary so the route does not scrape through the state.
    if node_core.any():
        dist_from_node = cv2.distanceTransform(
            (1 - node_core.astype(np.uint8)).astype(np.uint8),
            cv2.DIST_L2,
            3,
        )
        near_node = dist_from_node < 14
        cost[near_node] *= 3.5

    cost[~allowed] = 1e9

    p1 = weighted_dijkstra_path(cost, allowed, start, mid)
    p2 = weighted_dijkstra_path(cost, allowed, mid, goal)

    if not p1 or not p2:
        return None, {
            "weighted_midpoint_used": False,
            "reason": "weighted_tail_mid_or_mid_head_failed",
        }

    return p1 + p2[1:], {
        "weighted_midpoint_used": True,
        "reason": "passed",
    }


def route_self_loop_with_one_guide(
    image,
    arrow_ann,
    text_annotations,
    node_annotations,
    tail,
    head,
    loop_node,
    guide_points,
    side_name,
    node_ellipse_margin=10,
    corridor_thickness=75,
    pad=12,
    midpoint_exclusion_radius=18,
):
    h, w = image.shape[:2]

    ax, ay, aw, ah = arrow_ann["bbox"]

    xs = [ax, ax + aw, tail["x"], head["x"]] + [p[0] for p in guide_points]
    ys = [ay, ay + ah, tail["y"], head["y"]] + [p[1] for p in guide_points]

    x1p = clamp(int(min(xs)) - corridor_thickness - pad, 0, w - 1)
    y1p = clamp(int(min(ys)) - corridor_thickness - pad, 0, h - 1)
    x2p = clamp(int(max(xs)) + corridor_thickness + pad, 0, w - 1)
    y2p = clamp(int(max(ys)) + corridor_thickness + pad, 0, h - 1)

    crop = image[y1p:y2p + 1, x1p:x2p + 1]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    ch, cw = gray.shape

    skel = binarize_and_skeletonize_for_routing(gray)

    local_guide = localize_points(guide_points, x1p, y1p)

    guide_band = draw_polyline_mask(
        (ch, cw),
        local_guide,
        thickness=corridor_thickness,
    )

    arrow_region = np.zeros((ch, cw), dtype=bool)

    lx1 = clamp(ax - x1p, 0, cw - 1)
    ly1 = clamp(ay - y1p, 0, ch - 1)
    lx2 = clamp(ax + aw - x1p, 0, cw - 1)
    ly2 = clamp(ay + ah - y1p, 0, ch - 1)

    arrow_region[ly1:ly2 + 1, lx1:lx2 + 1] = True

    allowed_region = guide_band | arrow_region

    node_core = ellipse_mask_for_bbox(
        (ch, cw),
        loop_node["bbox"],
        x1p,
        y1p,
        margin=node_ellipse_margin,
    )

    text_block = np.zeros((ch, cw), dtype=bool)

    for ann in text_annotations:
        if not bbox_intersects(ann["bbox"], arrow_ann["bbox"]):
            continue

        bx, by, bw, bh = ann["bbox"]

        tx1 = clamp(bx - x1p - 3, 0, cw - 1)
        ty1 = clamp(by - y1p - 3, 0, ch - 1)
        tx2 = clamp(bx + bw - x1p + 3, 0, cw - 1)
        ty2 = clamp(by + bh - y1p + 3, 0, ch - 1)

        text_block[ty1:ty2 + 1, tx1:tx2 + 1] = True

    route_mask = skel & allowed_region & (~node_core) & (~text_block)

    local_tail = (
        int(round(tail["x"] - x1p)),
        int(round(tail["y"] - y1p)),
    )

    local_head = (
        int(round(head["x"] - x1p)),
        int(round(head["y"] - y1p)),
    )

    # Prefer skeleton points for start/goal, but allow weighted fallback later.
    start = nearest_mask_pixel(route_mask, local_tail, max_radius=85)
    goal = nearest_mask_pixel(route_mask, local_head, max_radius=85)

    weighted_allowed = allowed_region & (~node_core) & (~text_block)

    if start is None:
        start = nearest_mask_pixel(weighted_allowed, local_tail, max_radius=85)

    if goal is None:
        goal = nearest_mask_pixel(weighted_allowed, local_head, max_radius=85)

    if start is None or goal is None:
        return None, {
            "side": side_name,
            "reason": "could_not_snap_tail_or_head",
        }

    if len(local_guide) >= 4:
        synthetic_mid_target = np.mean(
            np.array(local_guide[1:-1], dtype=np.float32),
            axis=0,
        )
    else:
        synthetic_mid_target = np.mean(
            np.array(local_guide, dtype=np.float32),
            axis=0,
        )

    mid = choose_midpoint_on_real_loop_ink(
        route_mask=route_mask,
        guide_band=guide_band,
        arrow_region=arrow_region,
        start=start,
        goal=goal,
        synthetic_mid_target=synthetic_mid_target,
        node_bbox=loop_node["bbox"],
        crop_origin=(x1p, y1p),
        endpoint_exclusion_radius=midpoint_exclusion_radius,
    )

    if mid is None:
        # Relax midpoint candidate only, not the final logic.
        relaxed_for_mid = dilate_bool_mask(route_mask, k=3, iterations=1)
        relaxed_for_mid &= allowed_region
        relaxed_for_mid &= ~node_core
        relaxed_for_mid &= ~text_block

        mid = choose_midpoint_on_real_loop_ink(
            route_mask=relaxed_for_mid,
            guide_band=guide_band,
            arrow_region=arrow_region,
            start=start,
            goal=goal,
            synthetic_mid_target=synthetic_mid_target,
            node_bbox=loop_node["bbox"],
            crop_origin=(x1p, y1p),
            endpoint_exclusion_radius=midpoint_exclusion_radius,
        )

    if mid is None:
        return None, {
            "side": side_name,
            "reason": "could_not_find_midpoint_on_loop_ink",
        }

    # First try strict skeleton route, exactly like manual tail-mid-head annotation.
    path1 = shortest_path_on_binary_mask(route_mask, start, mid)
    path2 = shortest_path_on_binary_mask(route_mask, mid, goal)

    route_method = "skeleton_tail_mid_head"
    relaxed_used = False

    if not path1 or not path2:
        relaxed_mask = dilate_bool_mask(route_mask, k=3, iterations=1)
        relaxed_mask &= allowed_region
        relaxed_mask &= ~node_core
        relaxed_mask &= ~text_block

        start_r = nearest_mask_pixel(relaxed_mask, local_tail, max_radius=85)
        goal_r = nearest_mask_pixel(relaxed_mask, local_head, max_radius=85)
        mid_r = nearest_mask_pixel(relaxed_mask & guide_band, mid, max_radius=30)

        if start_r is not None and goal_r is not None and mid_r is not None:
            path1 = shortest_path_on_binary_mask(relaxed_mask, start_r, mid_r)
            path2 = shortest_path_on_binary_mask(relaxed_mask, mid_r, goal_r)

            if path1 and path2:
                start, mid, goal = start_r, mid_r, goal_r
                relaxed_used = True
                route_method = "relaxed_skeleton_tail_mid_head"

    # New stubborn-loop fallback:
    # still forced through the real midpoint, but not limited to 1-pixel skeleton.
    if not path1 or not path2:
        weighted_path, weighted_debug = weighted_route_forced_through_midpoint(
            gray=gray,
            allowed_region=allowed_region,
            node_core=node_core,
            local_guide=local_guide,
            start=start,
            mid=mid,
            goal=goal,
            extra_block_mask=text_block,
        )

        if weighted_path is None:
            return None, {
                "side": side_name,
                "reason": "tail_mid_or_mid_head_path_failed",
                "weighted_debug": weighted_debug,
            }

        full_path = weighted_path
        route_method = "weighted_tail_mid_head"
    else:
        full_path = path1 + path2[1:]

    out_path = [
        pt(x + x1p, y + y1p)
        for x, y in full_path
    ]

    ratio = path_inside_node_ratio_strict(
        out_path,
        loop_node["bbox"],
        margin=10,
    )

    chord = math.hypot(tail["x"] - head["x"], tail["y"] - head["y"])
    loop_len = path_length_pixels(out_path)
    min_reasonable_len = max(25.0, 1.25 * chord)

    if ratio > 0.03:
        return None, {
            "side": side_name,
            "reason": "path_still_enters_node",
            "inside_node_ratio": ratio,
            "path_length": loop_len,
            "min_reasonable_len": min_reasonable_len,
            "relaxed_used": relaxed_used,
            "route_method": route_method,
        }

    if loop_len < min_reasonable_len:
        return None, {
            "side": side_name,
            "reason": "path_too_short_for_loop",
            "inside_node_ratio": ratio,
            "path_length": loop_len,
            "min_reasonable_len": min_reasonable_len,
            "relaxed_used": relaxed_used,
            "route_method": route_method,
        }

    return out_path, {
        "side": side_name,
        "reason": "passed",
        "inside_node_ratio": ratio,
        "path_length": loop_len,
        "min_reasonable_len": min_reasonable_len,
        "relaxed_used": relaxed_used,
        "route_method": route_method,
        "midpoint": {
            "x": int(mid[0] + x1p),
            "y": int(mid[1] + y1p),
        },
    }



def extract_fa_self_loop_path_via_midpoint(
    image,
    arrow_ann,
    text_annotations,
    node_annotations,
    tail,
    head,
    node_ellipse_margin=10,
    corridor_thickness=75,
    pad=12,
    midpoint_exclusion_radius=18,
):
    """
    Self-loop router inspired by the manual GUI.

    It is applied to EVERY FA self-loop, not just one loop:
    - try a primary synthetic side based on geometry;
    - if it fails, try top/right/bottom/left;
    - for each side, find a real midpoint on black ink;
    - route tail -> midpoint -> head on skeleton.
    """

    is_loop, prev_node, next_node = is_self_loop_arrow(arrow_ann, node_annotations)

    if not is_loop:
        return None, {
            "midpoint_router_used": False,
            "reason": "not_self_loop",
        }

    if prev_node is None:
        return None, {
            "midpoint_router_used": False,
            "reason": "missing_loop_node",
        }

    side_order = ordered_loop_sides_from_geometry(arrow_ann, prev_node)

    attempts = []
    best_path = None
    best_debug = None
    best_score = float("inf")

    for side in side_order:
        guide_points = synthetic_fa_guide_points_for_side(
            arrow_ann,
            tail,
            head,
            side=side,
        )

        path, debug = route_self_loop_with_one_guide(
            image=image,
            arrow_ann=arrow_ann,
            text_annotations=text_annotations,
            node_annotations=node_annotations,
            tail=tail,
            head=head,
            loop_node=prev_node,
            guide_points=guide_points,
            side_name=side,
            node_ellipse_margin=node_ellipse_margin,
            corridor_thickness=corridor_thickness,
            pad=pad,
            midpoint_exclusion_radius=midpoint_exclusion_radius,
        )

        attempts.append(debug)

        if path is not None:
            # Prefer low inside ratio and long actual loop path.
            ratio = debug.get("inside_node_ratio", 1.0)
            plen = debug.get("path_length", 0.0)
            score = ratio - 0.0001 * plen

            if score < best_score:
                best_score = score
                best_path = path
                best_debug = debug

            # First successful side is usually correct because side_order is geometry-prioritized.
            return path, {
                "midpoint_router_used": True,
                "reason": "passed",
                "selected_side": side,
                "inside_node_ratio": ratio,
                "path_length": plen,
                "midpoint": debug.get("midpoint"),
                "attempts": attempts,
            }

    if best_path is not None:
        return best_path, {
            "midpoint_router_used": True,
            "reason": "best_available_path_used",
            "selected_side": best_debug.get("side"),
            "inside_node_ratio": best_debug.get("inside_node_ratio"),
            "path_length": best_debug.get("path_length"),
            "midpoint": best_debug.get("midpoint"),
            "attempts": attempts,
        }

    return None, {
        "midpoint_router_used": False,
        "reason": "all_sides_failed",
        "attempts": attempts,
    }

def extract_fa_path(
    image,
    arrow_ann,
    text_annotations,
    node_annotations,
    tail,
    head,
    pad=12,
    ink_threshold=180,
    endpoint_keep_radius=38,
    node_ellipse_margin=10,
    text_forbid_margin=4,
    corridor_thickness=65,
    allow_white_bridge=True,
):
    h, w = image.shape[:2]

    prev_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_prev"))
    next_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_next"))

    guide_points = synthetic_fa_guide_points(
        arrow_ann,
        tail,
        head,
        prev_node,
        next_node,
    )

    ax, ay, aw, ah = arrow_ann["bbox"]

    xs = [ax, ax + aw, tail["x"], head["x"]] + [p[0] for p in guide_points]
    ys = [ay, ay + ah, tail["y"], head["y"]] + [p[1] for p in guide_points]

    x1p = clamp(int(min(xs)) - pad - corridor_thickness, 0, w - 1)
    y1p = clamp(int(min(ys)) - pad - corridor_thickness, 0, h - 1)
    x2p = clamp(int(max(xs)) + pad + corridor_thickness, 0, w - 1)
    y2p = clamp(int(max(ys)) + pad + corridor_thickness, 0, h - 1)

    crop = image[y1p:y2p + 1, x1p:x2p + 1]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    ch, cw = gray.shape
    local_guide = localize_points(guide_points, x1p, y1p)

    corridor = draw_polyline_mask(
        (ch, cw),
        local_guide,
        thickness=corridor_thickness,
    )

    arrow_region = np.zeros((ch, cw), dtype=bool)

    lx1 = clamp(ax - x1p, 0, cw - 1)
    ly1 = clamp(ay - y1p, 0, ch - 1)
    lx2 = clamp(ax + aw - x1p, 0, cw - 1)
    ly2 = clamp(ay + ah - y1p, 0, ch - 1)

    arrow_region[ly1:ly2 + 1, lx1:lx2 + 1] = True

    allowed = corridor | arrow_region

    ink = gray < ink_threshold
    ink_u8 = ink.astype(np.uint8) * 255
    kernel = np.ones((3, 3), np.uint8)
    ink_u8 = cv2.morphologyEx(ink_u8, cv2.MORPH_CLOSE, kernel, iterations=1)
    ink = ink_u8 > 0

    for ann in text_annotations:
        if not bbox_intersects(ann["bbox"], arrow_ann["bbox"]):
            continue

        bx, by, bw, bh = ann["bbox"]

        lx1 = clamp(bx - x1p - text_forbid_margin, 0, cw - 1)
        ly1 = clamp(by - y1p - text_forbid_margin, 0, ch - 1)
        lx2 = clamp(bx + bw - x1p + text_forbid_margin, 0, cw - 1)
        ly2 = clamp(by + bh - y1p + text_forbid_margin, 0, ch - 1)

        allowed[ly1:ly2 + 1, lx1:lx2 + 1] = False

    node_forbidden = np.zeros((ch, cw), dtype=bool)

    for ann in node_annotations:
        if not bbox_intersects(ann["bbox"], arrow_ann["bbox"]):
            continue

        emask = ellipse_mask_for_bbox(
            (ch, cw),
            ann["bbox"],
            x1p,
            y1p,
            margin=node_ellipse_margin,
        )

        allowed[emask] = False
        node_forbidden |= emask

    local_tail = (
        int(round(tail["x"] - x1p)),
        int(round(tail["y"] - y1p)),
    )

    local_head = (
        int(round(head["x"] - x1p)),
        int(round(head["y"] - y1p)),
    )

    is_loop, loop_node, _ = is_self_loop_arrow(arrow_ann, node_annotations)

    effective_endpoint_keep_radius = endpoint_keep_radius

    if is_loop:
        effective_endpoint_keep_radius = adaptive_endpoint_radius_for_loop(
            tail,
            head,
            endpoint_keep_radius,
        )

    for cx, cy in [local_tail, local_head]:
        cx = clamp(cx, 0, cw - 1)
        cy = clamp(cy, 0, ch - 1)

        disk = make_disk_mask((ch, cw), cx, cy, effective_endpoint_keep_radius)
        allowed[disk] = True

    if is_loop and loop_node is not None:
        # Re-block deep node interior after endpoint restoration.
        # This prevents close tail/head restore disks from making a tunnel through the node.
        node_core = ellipse_mask_for_bbox(
            (ch, cw),
            loop_node["bbox"],
            x1p,
            y1p,
            margin=-6,
        )
        allowed[node_core] = False

    if allow_white_bridge:
        ink_u8 = ink.astype(np.uint8)

        dist_to_ink = cv2.distanceTransform(
            (1 - ink_u8).astype(np.uint8),
            cv2.DIST_L2,
            3,
        )

        cost = 1.0 + 7.5 * dist_to_ink
        cost[ink] = 1.0
    else:
        allowed &= ink
        cost = np.ones((ch, cw), dtype=np.float32)

    guide_line = draw_polyline_mask(
        (ch, cw),
        local_guide,
        thickness=3,
    ).astype(np.uint8)

    dist_to_guide = cv2.distanceTransform(
        (1 - guide_line).astype(np.uint8),
        cv2.DIST_L2,
        3,
    )

    cost += 0.55 * dist_to_guide

    if node_forbidden.any():
        dist_from_node = cv2.distanceTransform(
            (1 - node_forbidden.astype(np.uint8)).astype(np.uint8),
            cv2.DIST_L2,
            3,
        )

        near_node = dist_from_node < 20
        cost[near_node] *= 4.5

    cost[~allowed] = 1e9

    start = nearest_allowed_pixel(allowed, local_tail, max_radius=75)
    goal = nearest_allowed_pixel(allowed, local_head, max_radius=75)

    if start is None or goal is None:
        return straight_line_path(tail, head)

    path_xy = weighted_dijkstra_path(cost, allowed, start, goal)

    if not path_xy or len(path_xy) < 2:
        return straight_line_path(tail, head)

    return [pt(x + x1p, y + y1p) for x, y in path_xy]


def extract_fa_path_adaptive(
    image,
    arrow_ann,
    text_annotations,
    node_annotations,
    tail,
    head,
    bad_ratio_threshold=0.03,
):
    prev_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_prev"))
    next_node = find_node_by_source_id(node_annotations, arrow_ann.get("arrow_next"))

    is_self_loop = is_same_node(prev_node, next_node)

    if not is_self_loop:
        path = extract_fa_path(
            image=image,
            arrow_ann=arrow_ann,
            text_annotations=text_annotations,
            node_annotations=node_annotations,
            tail=tail,
            head=head,
        )

        return path, {
            "adaptive": False,
            "self_loop": False,
            "method": "standard_fa_router",
            "inside_node_ratio": None,
            "selected_config": None,
            "passed": None,
        }

    midpoint_path, midpoint_debug = extract_fa_self_loop_path_via_midpoint(
        image=image,
        arrow_ann=arrow_ann,
        text_annotations=text_annotations,
        node_annotations=node_annotations,
        tail=tail,
        head=head,
        node_ellipse_margin=10,
        corridor_thickness=75,
    )

    if midpoint_path is not None:
        return midpoint_path, {
            "adaptive": True,
            "self_loop": True,
            "method": "midpoint_skeleton_router",
            "inside_node_ratio": midpoint_debug.get("inside_node_ratio"),
            "selected_config": {
                "method": "synthetic_guide_to_real_midpoint",
                "node_ellipse_margin": 10,
                "corridor_thickness": 75,
            },
            "passed": True,
            "midpoint_debug": midpoint_debug,
        }

    best_path = None
    best_ratio = float("inf")
    best_cfg = None

    tried = [{
        "method": "midpoint_skeleton_router",
        "debug": midpoint_debug,
    }]

    for cfg in FA_RETRY_CONFIGS:
        path = extract_fa_path(
            image=image,
            arrow_ann=arrow_ann,
            text_annotations=text_annotations,
            node_annotations=node_annotations,
            tail=tail,
            head=head,
            **cfg,
        )

        ratio = path_inside_node_ratio_strict(
            path=path,
            node_bbox=prev_node["bbox"],
            margin=10,
        )

        tried.append({
            "method": "weighted_retry_router",
            "config": cfg,
            "inside_node_ratio": ratio,
        })

        if ratio < best_ratio:
            best_ratio = ratio
            best_path = path
            best_cfg = cfg

        if ratio <= bad_ratio_threshold:
            return path, {
                "adaptive": True,
                "self_loop": True,
                "method": "weighted_retry_router",
                "inside_node_ratio": ratio,
                "selected_config": cfg,
                "passed": True,
                "tried": tried,
            }

    return best_path, {
        "adaptive": True,
        "self_loop": True,
        "method": "best_weighted_retry_router",
        "inside_node_ratio": best_ratio,
        "selected_config": best_cfg,
        "passed": False,
        "tried": tried,
    }


def extract_arrow_path(
    image,
    arrow_ann,
    text_annotations,
    node_annotations,
    tail,
    head,
    routing_mode="auto",
    return_debug=False,
):
    if routing_mode == "auto":
        routing_mode = (
            "flowchart"
            if arrow_ann.get("waypoints") and len(arrow_ann.get("waypoints", [])) >= 2
            else "fa"
        )

    if routing_mode == "flowchart":
        path = extract_flowchart_path(
            image,
            arrow_ann,
            node_annotations + text_annotations,
            tail,
            head,
        )
        debug = {"routing_mode_used": "flowchart"}

    elif routing_mode == "fa":
        path, fa_debug = extract_fa_path_adaptive(
            image=image,
            arrow_ann=arrow_ann,
            text_annotations=text_annotations,
            node_annotations=node_annotations,
            tail=tail,
            head=head,
        )
        debug = {"routing_mode_used": "fa", "fa_adaptive_debug": fa_debug}

    elif routing_mode == "straight":
        path = straight_line_path(tail, head)
        debug = {"routing_mode_used": "straight"}

    elif routing_mode == "waypoint":
        path = waypoint_path(arrow_ann, tail, head)
        debug = {"routing_mode_used": "waypoint"}

    else:
        raise ValueError(f"Unknown routing mode: {routing_mode}")

    if return_debug:
        return path, debug

    return path


## 5. Optional overlays and node-link resolution

Overlays are diagnostic only. Node-link resolution first uses the source
annotation mapping and then applies a geometric fallback when an endpoint link
is absent.


In [ ]:
# ============================================================
# Overlay
# ============================================================

def draw_overlay(image, nodes, arrows):
    overlay = image.copy()

    for node in nodes:
        x, y, w, h = node["bbox"]

        cv2.rectangle(
            overlay,
            (x, y),
            (x + w, y + h),
            (0, 180, 0),
            2,
        )

        label = node.get("type", "node")
        cv2.putText(
            overlay,
            f'{label}:{node["id"]}',
            (x, max(15, y - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 150, 0),
            1,
            cv2.LINE_AA,
        )

    for arrow in arrows:
        path = arrow.get("path", [])

        if len(path) >= 2:
            pts = np.array(
                [[int(round(p["x"])), int(round(p["y"]))] for p in path],
                dtype=np.int32,
            )

            cv2.polylines(
                overlay,
                [pts],
                False,
                (0, 0, 255),
                2,
            )

        x, y, w, h = arrow["bbox"]

        cv2.rectangle(
            overlay,
            (x, y),
            (x + w, y + h),
            (255, 0, 0),
            1,
        )

        tail = arrow["tail"]
        head = arrow["head"]

        cv2.circle(
            overlay,
            (int(round(tail["x"])), int(round(tail["y"]))),
            5,
            (255, 0, 255),
            -1,
        )

        cv2.circle(
            overlay,
            (int(round(head["x"])), int(round(head["y"]))),
            5,
            (0, 255, 0),
            -1,
        )

        cv2.putText(
            overlay,
            f'Tail{arrow["id"]}',
            (int(round(tail["x"])) + 5, int(round(tail["y"])) - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 0, 255),
            1,
            cv2.LINE_AA,
        )

        cv2.putText(
            overlay,
            f'Head{arrow["id"]}',
            (int(round(head["x"])) + 5, int(round(head["y"])) - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 140, 0),
            1,
            cv2.LINE_AA,
        )

        for p in arrow.get("equidistant_points", []):
            cv2.putText(
                overlay,
                p["label"],
                (int(p["x"]), int(p["y"])),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.32,
                (30, 30, 30),
                1,
                cv2.LINE_AA,
            )

    return overlay


In [ ]:
# ============================================================
# Node-linking fallback
# ============================================================

def point_to_bbox_distance(p, bbox):
    """
    Distance from point to an axis-aligned bbox.
    Returns 0 if point lies inside the bbox.
    Coordinates must be in the same space.
    """
    x, y, w, h = bbox
    px = float(p["x"])
    py = float(p["y"])

    dx = max(x - px, 0.0, px - (x + w))
    dy = max(y - py, 0.0, py - (y + h))

    return math.hypot(dx, dy)


def node_snap_threshold(node_anns):
    """
    Adaptive threshold in original image/JSON coordinate space.
    This avoids hardcoding one threshold for all datasets/resolutions.
    """
    if not node_anns:
        return 0.0

    diags = []
    for ann in node_anns:
        x, y, w, h = ann["bbox"]
        diags.append(math.hypot(w, h))

    med = float(np.median(diags)) if diags else 0.0
    return max(12.0, min(70.0, 0.35 * med))


def infer_node_id_from_candidates(candidates, node_records, max_dist):
    """
    Geometric fallback for missing tail_node_id/head_node_id.

    It snaps an endpoint to the nearest node only if the endpoint is close
    enough to that node bbox. This preserves legitimate start arrows, whose
    free tail is usually far from every node.
    """
    best = None

    for cand_name, p in candidates:
        if p is None:
            continue

        for rec in node_records:
            d = point_to_bbox_distance(p, rec["raw_bbox"])

            if best is None or d < best["distance"]:
                best = {
                    "node_id": rec["new_id"],
                    "distance": float(d),
                    "candidate": cand_name,
                    "point": {"x": float(p["x"]), "y": float(p["y"])},
                }

    if best is None:
        return None, {
            "used": False,
            "reason": "no_candidate_or_node",
            "max_dist": max_dist,
        }

    if best["distance"] <= max_dist:
        return best["node_id"], {
            "used": True,
            "reason": "snapped_to_nearest_node",
            "max_dist": max_dist,
            **best,
        }

    return None, {
        "used": False,
        "reason": "nearest_node_too_far",
        "max_dist": max_dist,
        "nearest": best,
    }


def resolve_arrow_node_ids(
    ann,
    tail,
    head,
    raw_path,
    node_id_map,
    node_records,
    snap_dist,
):
    """
    First trust the original annotation ids. If they fail, use geometric
    endpoint snapping. This fixes cases where train/test had matching ids
    but val used ids that did not map into node_id_map.
    """
    arrow_prev = ann.get("arrow_prev", None)
    arrow_next = ann.get("arrow_next", None)

    tail_node_id = node_id_map.get(arrow_prev, None)
    head_node_id = node_id_map.get(arrow_next, None)

    debug = {
        "arrow_prev": arrow_prev,
        "arrow_next": arrow_next,
        "tail_from_annotation": tail_node_id is not None,
        "head_from_annotation": head_node_id is not None,
        "snap_dist": snap_dist,
    }

    first_path = raw_path[0] if raw_path else None
    last_path = raw_path[-1] if raw_path else None

    if tail_node_id is None:
        tail_candidates = [
            ("tail_hint", tail),
            ("path_start", first_path),
        ]

        tail_node_id, tail_debug = infer_node_id_from_candidates(
            tail_candidates,
            node_records,
            snap_dist,
        )
        debug["tail_fallback"] = tail_debug
    else:
        debug["tail_fallback"] = {"used": False, "reason": "annotation_mapping_success"}

    if head_node_id is None:
        head_candidates = [
            ("head_hint", head),
            ("path_end", last_path),
        ]

        head_node_id, head_debug = infer_node_id_from_candidates(
            head_candidates,
            node_records,
            snap_dist,
        )
        debug["head_fallback"] = head_debug
    else:
        debug["head_fallback"] = {"used": False, "reason": "annotation_mapping_success"}

    return tail_node_id, head_node_id, debug


## 6. Per-image conversion

Each source image produces one JSON file with:

- `size`: the common target canvas;
- `nodes`: sequential node identifiers and scaled bounding boxes;
- `arrows`: scaled endpoint hints, ordered paths, and source/target node IDs.

The source image itself is read for routing but is not redistributed or copied
by this notebook.


In [ ]:
# ============================================================
# Per-image conversion
# ============================================================

def convert_one_image(
    image_info,
    annotations,
    image_dir,
    out_json_dir,
    out_overlay_dir,
    save_overlay=True,
    trace_from_image=True,
    mask_w=1024,
    mask_h=1024,
    routing_mode="auto",
    save_debug=True,
):
    file_name = image_info["file_name"]
    image_path = image_dir / file_name

    if not image_path.exists():
        return {"image": file_name, "status": "missing_image", "arrows": 0, "nodes": 0}

    image = cv2.imread(str(image_path))

    if image is None:
        return {"image": file_name, "status": "unreadable_image", "arrows": 0, "nodes": 0}

    json_w = int(image_info.get("width", image.shape[1]))
    json_h = int(image_info.get("height", image.shape[0]))

    sx = mask_w / json_w
    sy = mask_h / json_h

    arrow_anns = [a for a in annotations if a.get("category") == "arrow"]
    text_anns = [a for a in annotations if a.get("category") == "text"]
    node_anns = [a for a in annotations if a.get("category") not in {"arrow", "text"}]

    node_id_map = {}
    nodes = []
    overlay_nodes = []
    node_records = []

    for idx, ann in enumerate(node_anns, start=1):
        new_id = f"N{idx}"

        source_id = ann.get("id")
        source_aid = ann.get("aid")

        if source_id is not None:
            node_id_map[source_id] = new_id

        if source_aid is not None:
            node_id_map[source_aid] = new_id

        nodes.append({"id": new_id, "bbox": scale_bbox(ann["bbox"], sx, sy)})

        overlay_nodes.append({
            "id": new_id,
            "type": ann.get("category", "node"),
            "bbox": [int(round(v)) for v in ann["bbox"]],
        })

        node_records.append({
            "new_id": new_id,
            "raw_bbox": ann["bbox"],
            "source_id": source_id,
            "source_aid": source_aid,
            "type": ann.get("category", "node"),
        })

    snap_dist = node_snap_threshold(node_anns)

    arrows = []
    overlay_arrows = []

    for idx, ann in enumerate(arrow_anns, start=1):
        k = ann.get("keypoints", [])

        if len(k) < 6:
            continue

        tail = pt(k[0], k[1])
        head = pt(k[3], k[4])

        if trace_from_image:
            raw_path, routing_debug = extract_arrow_path(
                image=image,
                arrow_ann=ann,
                text_annotations=text_anns,
                node_annotations=node_anns,
                tail=tail,
                head=head,
                routing_mode=routing_mode,
                return_debug=True,
            )
        else:
            raw_path = waypoint_path(ann, tail, head)
            routing_debug = {"routing_mode_used": "waypoint_or_straight_no_trace"}

        tail_scaled = scale_point(tail, sx, sy)
        head_scaled = scale_point(head, sx, sy)
        path_scaled = scale_path(raw_path, sx, sy)

        tail_node_id, head_node_id, node_link_debug = resolve_arrow_node_ids(
            ann=ann,
            tail=tail,
            head=head,
            raw_path=raw_path,
            node_id_map=node_id_map,
            node_records=node_records,
            snap_dist=snap_dist,
        )

        arrow_json = {
            "id": idx,
            "tail_hint": tail_scaled,
            "head_hint": head_scaled,
            "path": path_scaled,
            "tail_node_id": tail_node_id,
            "head_node_id": head_node_id,
            "scale_debug": {
                "json_space": {"w": json_w, "h": json_h},
                "mask_space": {"w": mask_w, "h": mask_h},
                "sx": sx,
                "sy": sy,
                "sx_arrow": sx,
                "sy_arrow": sy,
                "routing_mode_requested": routing_mode,
            },
            "control_points": [tail_scaled, head_scaled],
        }

        if save_debug:
            arrow_json["routing_debug"] = routing_debug

        arrows.append(arrow_json)

        overlay_arrows.append({
            "id": idx,
            "bbox": [int(round(v)) for v in ann["bbox"]],
            "tail": tail,
            "head": head,
            "path": raw_path,
            "equidistant_points": sample_equidistant_points(raw_path, count=40),
        })

    out = {
        "image": file_name,
        "size": {"w": mask_w, "h": mask_h},
        "nodes": nodes,
        "arrows": arrows,
    }

    out_json_dir.mkdir(parents=True, exist_ok=True)
    out_json_path = out_json_dir / f"{Path(file_name).stem}.json"

    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)

    if save_overlay:
        out_overlay_dir.mkdir(parents=True, exist_ok=True)
        overlay = draw_overlay(image.copy(), overlay_nodes, overlay_arrows)
        overlay_path = out_overlay_dir / f"{Path(file_name).stem}_overlay.png"
        cv2.imwrite(str(overlay_path), overlay)

    return {"image": file_name, "status": "ok", "arrows": len(arrows), "nodes": len(nodes)}


## 7. Split processing and automatic discovery

The notebook uses thread-based parallelism because it is portable across common
Jupyter environments. Every image is processed independently, so the converted
records are unchanged by worker count.


## 8. Path validation and output checks

The final validation reports coordinate-bound violations and unlinked endpoints.
Unlinked endpoints are reported rather than automatically treated as failures,
because some source annotations may legitimately lack one attachment.


In [ ]:

# ============================================================
# Repository-level validation
# ============================================================

def require_nonempty_path(value: str, variable_name: str) -> Path:
    value = str(value).strip()
    if not value:
        raise ValueError(
            f"{variable_name} is blank. Fill it in the USER SETTINGS cell before running."
        )
    return Path(value).expanduser().resolve()


def annotation_bounds_report(json_path: Path) -> Dict[str, int]:
    """Validate one converted annotation without changing it."""
    with json_path.open("r", encoding="utf-8") as f:
        record = json.load(f)

    width = int(record.get("size", {}).get("w", 0))
    height = int(record.get("size", {}).get("h", 0))

    report = {
        "files": 1,
        "nodes": 0,
        "arrows": 0,
        "invalid_canvas": 0,
        "out_of_bounds_nodes": 0,
        "out_of_bounds_endpoints": 0,
        "out_of_bounds_path_points": 0,
        "unlinked_tail_ids": 0,
        "unlinked_head_ids": 0,
    }

    if width != TARGET_WIDTH or height != TARGET_HEIGHT:
        report["invalid_canvas"] += 1

    for node in record.get("nodes", []):
        report["nodes"] += 1
        x, y, w, h = node["bbox"]
        if x < 0 or y < 0 or w < 0 or h < 0 or x + w > width or y + h > height:
            report["out_of_bounds_nodes"] += 1

    for arrow in record.get("arrows", []):
        report["arrows"] += 1

        for key in ("tail_hint", "head_hint"):
            point = arrow.get(key)
            if point is None:
                report["out_of_bounds_endpoints"] += 1
                continue
            x, y = float(point["x"]), float(point["y"])
            if x < 0 or y < 0 or x >= width or y >= height:
                report["out_of_bounds_endpoints"] += 1

        for point in arrow.get("path", []):
            x, y = float(point["x"]), float(point["y"])
            if x < 0 or y < 0 or x >= width or y >= height:
                report["out_of_bounds_path_points"] += 1

        if arrow.get("tail_node_id") is None:
            report["unlinked_tail_ids"] += 1
        if arrow.get("head_node_id") is None:
            report["unlinked_head_ids"] += 1

    return report


def merge_count_reports(reports: Iterable[Dict[str, int]]) -> Dict[str, int]:
    merged: Dict[str, int] = defaultdict(int)
    for report in reports:
        for key, value in report.items():
            merged[key] += int(value)
    return dict(merged)


def validate_converted_repository(output_root: Path) -> Dict[str, object]:
    json_files = sorted(output_root.glob("*/*/json/*.json"))
    count_report = merge_count_reports(
        annotation_bounds_report(path) for path in json_files
    )

    split_counts: Dict[str, Dict[str, int]] = defaultdict(dict)
    for family_dir in sorted(p for p in output_root.iterdir() if p.is_dir()):
        for split_dir in sorted(p for p in family_dir.iterdir() if p.is_dir()):
            json_dir = split_dir / "json"
            if json_dir.exists():
                split_counts[family_dir.name][split_dir.name] = len(
                    list(json_dir.glob("*.json"))
                )

    return {
        "target_size": {"w": TARGET_WIDTH, "h": TARGET_HEIGHT},
        "families": dict(split_counts),
        "counts": count_report,
    }


## 9. Run conversion

Fill the two paths in the first code cell, then run this cell. The notebook
processes all selected families and writes a final
`conversion_validation_report.json`.


In [ ]:

# ============================================================
# RUN THE CONVERSION
# ============================================================

source_root = require_nonempty_path(SOURCE_DATASETS_ROOT, "SOURCE_DATASETS_ROOT")
output_root = require_nonempty_path(OUTPUT_ANNOTATIONS_ROOT, "OUTPUT_ANNOTATIONS_ROOT")

if source_root == output_root:
    raise ValueError("SOURCE_DATASETS_ROOT and OUTPUT_ANNOTATIONS_ROOT must be different.")

if ROUTING_MODE not in {"auto", "flowchart", "fa", "straight", "waypoint"}:
    raise ValueError(f"Unsupported ROUTING_MODE: {ROUTING_MODE}")

if TARGET_WIDTH <= 0 or TARGET_HEIGHT <= 0:
    raise ValueError("TARGET_WIDTH and TARGET_HEIGHT must be positive.")

workers = max(1, int(WORKERS))
output_root.mkdir(parents=True, exist_ok=True)

repository_results: Dict[str, object] = {}

for family in FAMILIES_TO_CONVERT:
    family_source = source_root / family
    family_output = output_root / family

    if not family_source.exists():
        raise FileNotFoundError(
            f"Dataset family '{family}' was not found at: {family_source}"
        )

    if SPLITS_TO_CONVERT is None:
        split_pairs = discover_json_splits(family_source)
    else:
        split_pairs = []
        for split_name in SPLITS_TO_CONVERT:
            image_dir = family_source / split_name
            json_path = family_source / f"{split_name}.json"

            if not image_dir.exists():
                print(f"Skipping {family}/{split_name}: missing image folder {image_dir}")
                continue
            if not json_path.exists():
                print(f"Skipping {family}/{split_name}: missing source JSON {json_path}")
                continue

            split_pairs.append((split_name, image_dir, json_path))

    if not split_pairs:
        raise RuntimeError(
            f"No valid split was found for '{family}'. Expected train.json with "
            "a matching train/ image folder, and similarly for other splits."
        )

    print("\n" + "=" * 72)
    print(f"DATASET FAMILY: {family}")
    print("=" * 72)
    for split_name, image_dir, json_path in split_pairs:
        print(f"  {split_name}: {json_path.name} -> {image_dir.name}/")

    family_results: Dict[str, object] = {}

    for split_name, image_dir, json_path in split_pairs:
        family_results[split_name] = process_split(
            split_name=split_name,
            image_dir=image_dir,
            json_path=json_path,
            out_root=family_output,
            workers=workers,
            save_overlay=SAVE_OVERLAYS,
            trace_from_image=TRACE_FROM_IMAGE,
            mask_w=TARGET_WIDTH,
            mask_h=TARGET_HEIGHT,
            routing_mode=ROUTING_MODE,
            save_debug=SAVE_ROUTING_DEBUG,
        )

    family_output.mkdir(parents=True, exist_ok=True)
    with (family_output / "all_splits_summary.json").open("w", encoding="utf-8") as f:
        json.dump(family_results, f, indent=2)

    repository_results[family] = family_results

with (output_root / "all_families_summary.json").open("w", encoding="utf-8") as f:
    json.dump(repository_results, f, indent=2)

validation_report = validate_converted_repository(output_root)
with (output_root / "conversion_validation_report.json").open("w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2)

print("\n" + "=" * 72)
print("CONVERSION COMPLETE")
print("=" * 72)
print(json.dumps(validation_report, indent=2))
print(
    "\nNext step: run the image-resizing notebook to create 1024 × 1024 "
    "images matching these annotation coordinates."
)
